# DINOv3 ConvNeXt pyramid decoder — batch inference

Runs the trained **frozen DINOv3 ConvNeXt backbone -> pyramid decoder ->
stride-4 peak heatmap -> local-max decode -> points + count** model over a
folder of images.

Uses `cropcounter` directly (`pip install -e .` from the repo root) — no
local helper functions here, `records_from_folder`, `predict_prob`,
`decode_classes`, `save_visualization` and `write_cvat_xml` all live in
`cropcounter.inference` and are imported below.

Defaults to the bundled anonymised val split (`../examples/data/val/images`,
6 images), so this runs out of the box once a checkpoint exists at
`../weights/decoder_best.pt`. Point `DATASETS` at your own folder(s) to run
over anything else.

**Outputs**, under `OUTPUT_ROOT/<tag>/` (`runs/inference/` by default,
gitignored):
- `counts.csv` — one row per image (`image_name, image_path, predicted_count`)
- `visualizations/<stem>.jpg` — decoded points overlay + predicted heatmap
- `predictions_cvat.xml` — CVAT-for-images-1.1 point export (re-importable into CVAT)

In [ ]:
%load_ext autoreload
%autoreload 2

import csv
import gc
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from cropcounter import (
    CropTileDataset, collate_val, decode_classes, load_checkpoint,
    parse_cvat_1_1, predict_prob, records_from_folder, resolve_device,
    save_visualization, write_cvat_xml,
)
from cropcounter.dinov3_pyramid import IMAGENET_MEAN, IMAGENET_STD
from cropcounter.inference import grid_hw

device = resolve_device()
print(torch.__version__, device)

# One colour per class for the inline overlays (class index -> colour).
class_colour = plt.get_cmap("tab10")

## Parameters

In [ ]:
# --- Checkpoint -------------------------------------------------------------
# Arrives from a training run (notebooks/training.ipynb) or is dropped in
# directly; this notebook doesn't require it to exist until this cell runs.
CKPT_PATH = Path("../weights/decoder_best.pt")

# --- Decode threshold (exposed knob; calibrate with notebooks/training.ipynb) -
# One value applies to every class; a multiclass checkpoint can take a
# per-class dict instead, e.g. {"Wheat": 0.35, "Beans": 0.25}.
TAU = 0.35

# --- Datasets to run (tag -> images/ folder) --------------------------------
DATASETS = {
    "examples_val": Path("../examples/demo/data/val/images"),
}

# --- Output location ---------------------------------------------------------
OUTPUT_ROOT = Path("runs/inference")

# --- Toggles ------------------------------------------------------------------
SAVE_VIZ    = True     # per-image points+heatmap .jpg
SAVE_XML    = True     # CVAT 1.1 point export per dataset
MAX_IMAGES  = None     # int to smoke-test on a subset; None = all images
NUM_WORKERS = 0        # bump up for a large folder; 0 keeps this demo simple/portable
PIN_MEMORY  = False
VIZ_MAX_SIDE = 1400    # downscale the imshow panel so big images don't balloon RAM

print("checkpoint:", CKPT_PATH.resolve())
print("output    :", OUTPUT_ROOT.resolve())
for _tag, _d in DATASETS.items():
    print(f"  {_tag}: {_d.resolve()}  exists={_d.exists()}")

## Load model

`load_checkpoint`'s `weights_dir=` override rebuilds the backbone from this notebook's own `../weights`, regardless of what working directory the checkpoint's saved config recorded.

In [ ]:
model, cfg = load_checkpoint(CKPT_PATH, device, weights_dir=Path("../weights"))
model.eval()

n_dec = sum(p.numel() for p in model.decoder.parameters())
print(f"loaded {CKPT_PATH.name} | backbone {cfg.backbone} | decoder {n_dec / 1e6:.2f}M params | {device}")
print(f"decode: stride {cfg.output_stride}, k {cfg.k}, nms_radius {cfg.nms_radius}, "
      f"sigma {cfg.sigma} | tau {TAU}")
print(f"classes ({cfg.n_classes}): {cfg.classes}")

## Run inference

Whole-image forward pass per image (fully convolutional, padded to a multiple of 32), decode, then write `counts.csv`, per-image visualizations, and the CVAT XML.

In [ ]:
def run_dataset(tag, images_dir):
    out_dir = OUTPUT_ROOT / tag
    viz_dir = out_dir / "visualizations"
    out_dir.mkdir(parents=True, exist_ok=True)
    if SAVE_VIZ:
        viz_dir.mkdir(parents=True, exist_ok=True)

    records = records_from_folder(images_dir, max_images=MAX_IMAGES)
    ds = CropTileDataset(records, images_dir, train=False,
                         output_stride=cfg.output_stride, sigma=cfg.sigma,
                         n_classes=cfg.n_classes)
    loader_kw = dict(batch_size=1, shuffle=False, collate_fn=collate_val,
                     num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    if NUM_WORKERS > 0:
        loader_kw["prefetch_factor"] = 2   # cap how many large tensors queue ahead
    loader = DataLoader(ds, **loader_kw)

    rows, image_preds = [], []
    for i, (rec, batch) in enumerate(zip(records, tqdm(loader, desc=f"{tag} ({len(records)} imgs)"))):
        assert batch["name"] == rec.name, "loader order drifted from records"
        w, h = rec.width, rec.height
        prob = predict_prob(model, batch["image"][0], device)          # (1, C, h, w)
        pts, scores, class_ids = decode_classes(
            prob, cfg.class_names, tau=TAU, k=cfg.k, nms_radius=cfg.nms_radius,
            output_stride=cfg.output_stride, width=w, height=h,
        )

        row = {"image_name": rec.name,
               "image_path": str((Path(images_dir) / rec.name).resolve()),
               "predicted_count": int(len(pts))}
        for c, name in enumerate(cfg.class_names):          # one column per class
            row[f"count_{name}"] = int((class_ids == c).sum())
        rows.append(row)
        image_preds.append({"name": rec.name, "width": w, "height": h,
                            "points": pts, "scores": scores,
                            "labels": [cfg.class_names[c] for c in class_ids]})
        if SAVE_VIZ:
            save_visualization(batch["image"][0], prob, pts, w, h,
                               viz_dir / f"{Path(rec.name).stem}.jpg",
                               output_stride=cfg.output_stride, max_side=VIZ_MAX_SIDE,
                               class_ids=class_ids, class_names=cfg.class_names)
        del prob, batch
        if (i + 1) % 200 == 0:
            gc.collect()   # release freed large-image tensors periodically

    fieldnames = ["image_name", "image_path", "predicted_count"] + [
        f"count_{name}" for name in cfg.class_names
    ]
    with open(out_dir / "counts.csv", "w", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    if SAVE_XML:
        write_cvat_xml(image_preds, out_dir / "predictions_cvat.xml")

    counts = np.array([r["predicted_count"] for r in rows])
    print(f"[{tag}] {len(rows)} images | total {counts.sum()} "
          f"mean {counts.mean():.1f} min {counts.min()} max {counts.max()} "
          f"| out: {out_dir.resolve()}")
    if cfg.n_classes > 1:
        print("        per class: " + ", ".join(
            f"{name} {sum(r[f'count_{name}'] for r in rows)}" for name in cfg.class_names))
    return rows, image_preds


all_results = {tag: run_dataset(tag, d) for tag, d in DATASETS.items()}

## Inline sanity check

Eyeball a few predictions and verify the CVAT XML round-trips, before trusting the batch outputs.

In [ ]:
sanity_tag = next(iter(DATASETS))
sanity_dir = DATASETS[sanity_tag]
sanity_recs = records_from_folder(sanity_dir)
sanity_ds = CropTileDataset(sanity_recs, sanity_dir, train=False,
                            output_stride=cfg.output_stride, sigma=cfg.sigma,
                            n_classes=cfg.n_classes)
n_show = min(3, len(sanity_ds))
picks = np.random.default_rng(0).choice(len(sanity_ds), size=n_show, replace=False)

fig, axes = plt.subplots(n_show, 2, figsize=(15, 20 * n_show / 3), squeeze=False)
for row, idx in enumerate(picks):
    rec = sanity_recs[int(idx)]
    item = sanity_ds[int(idx)]
    prob = predict_prob(model, item["image"], device)
    pts, _, class_ids = decode_classes(
        prob, cfg.class_names, tau=TAU, k=cfg.k, nms_radius=cfg.nms_radius,
        output_stride=cfg.output_stride, width=rec.width, height=rec.height,
    )
    img = item["image"].permute(1, 2, 0).numpy() * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    img = img[:rec.height, :rec.width].clip(0, 1)
    gh, gw = grid_hw(rec.width, rec.height, output_stride=cfg.output_stride)
    axes[row, 0].imshow(img)
    for c, name in enumerate(cfg.class_names):
        m = class_ids == c
        axes[row, 0].scatter(pts[m, 0], pts[m, 1], s=40, facecolors="none",
                             edgecolors=[class_colour(c)], label=f"{name}: {int(m.sum())}")
    axes[row, 0].legend(loc="upper right", fontsize=8)
    axes[row, 0].set_title(f"{rec.name[:60]}  pred {len(pts)}", fontsize=9)
    axes[row, 1].imshow(prob[0, :, :gh, :gw].max(0).values, cmap="hot", vmin=0, vmax=1)
    axes[row, 1].set_title("predicted heatmap" + (" (max over classes)" if cfg.n_classes > 1 else ""))
    for ax in axes[row]:
        ax.axis("off")

In [ ]:
# CVAT XML round-trip: parse the written file back and compare point totals.
xml_path = OUTPUT_ROOT / sanity_tag / "predictions_cvat.xml"
if xml_path.exists():
    parsed = parse_cvat_1_1(xml_path, labels=None)   # keep every class label
    total = sum(len(r.points) for r in parsed)
    print(f"round-trip: parsed {len(parsed)} images, {total} points from {xml_path.name}")
    per_label = {}
    for r in parsed:
        for pt in r.points:
            per_label[pt.label] = per_label.get(pt.label, 0) + 1
    print("  per label:", per_label)
else:
    print(f"no XML at {xml_path} (run the inference cell with SAVE_XML=True first)")